In [8]:
from urllib.request import urlopen
from deepagents import create_deep_agent
from deepagents.backends import StateBackend
from deepagents.backends.utils import create_file_data
from langchain_quickjs import CodeInterpreterMiddleware
from langgraph.checkpoint.memory import MemorySaver
from langchain_groq import ChatGroq 
from dotenv import load_dotenv
import os 

load_dotenv()

groq_api_key=os.getenv('GROQ_API_KEY')

model=ChatGroq(model='openai/gpt-oss-20b',groq_api_key=groq_api_key)
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.7', 'langchain': '1.4.3'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000023A4F144790>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000023A4F146E50>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [11]:
checkpointer=MemorySaver()
backend=StateBackend()

from pathlib import Path 
skills_content=Path('skills/langgraph/SKILL.md').read_text(encoding='utf-8')

skills_files={
    "/skills/langgraph/SKILL.md":create_file_data(skills_content),
}

In [14]:
from pathlib import Path
from deepagents.backends.utils import create_file_data

skill_dirs = ["langgraph", "python", "aws", "report_generator"]

skill_files = {
    f"/skills/{name}/SKILL.md": create_file_data(
        Path(f"skills/{name}/SKILL.md").read_text(encoding="utf-8")
    )
    for name in skill_dirs
}

In [15]:
skill_files

{'/skills/langgraph/SKILL.md': {'content': '# LangGraph Development Skill\n\n## Role\n\nYou are a LangGraph development specialist.\n\nYour responsibility is to design, implement, debug, and improve stateful AI agents and multi-step agent workflows using LangGraph.\n\nYou should build reliable, modular, observable, and maintainable agent workflows.\n\n---\n\n## Core Capabilities\n\nYou can work with:\n\n- LangGraph\n- LangChain Core messages\n- StateGraph\n- MessagesState\n- START and END\n- Nodes\n- Edges\n- Conditional edges\n- ToolNode\n- tools_condition\n- Checkpointers\n- MemorySaver\n- Human-in-the-loop\n- Interrupts\n- Command\n- Multi-agent workflows\n- Subgraphs\n- Streaming\n- Tool calling\n- Agent loops\n- LangSmith tracing\n- MCP-based tools\n\n---\n\n# 1. Graph Architecture\n\nUse LangGraph when the application requires:\n\n- Stateful workflows\n- Multiple steps\n- Tool execution\n- Agent loops\n- Human approval\n- Memory\n- Conditional routing\n- Multi-agent coordination\

## Creating DeepAgents

In [17]:
agent=create_deep_agent(
    model=model,
    backend=backend,
    skills=['/skills/'],
    checkpointer=checkpointer
)

result=agent.invoke(
    {
    'messages':[{'role':'user','content':'What skills do you have available, and when would you use each'}],
    'files':skill_files,
    },
    config={'configurable':{'thread_id':'skills-demo'}},


)
print(result['messages'][-1].content)

In [18]:
result=agent.invoke(
    {
    'messages':[{'role':'user','content':'How do I build a langraph with conditional routing'}],
    'files':skill_files,
    },
    config={'configurable':{'thread_id':'skills-demo'}},


)
print(result['messages'][-1].content)

for m in result['messages']:
    m.pretty_print()

Below is a practical, step‑by‑step guide (with a runnable example) that shows how to build a **LangGraph** that uses **conditional routing** to decide which tool or sub‑graph to invoke next.  
The example demonstrates the core concepts that the *LangGraph Development Skill* covers:  

* **StateGraph** – the graph container.  
* **Nodes** – functions or tool calls.  
* **Conditional edges** – `tools_condition` or a custom function that returns the next node name.  
* **State** – a dict that carries messages, memory, and any custom flags.  
* **Memory / Checkpointing** – optional, but shown for completeness.  
* **Human‑in‑the‑loop** – a quick “approval” node.  

Feel free to copy‑paste the code into a fresh Python file, install the dependencies (`langchain`, `langgraph`, `langsmith`), and run it.

---

## 1. Install the required packages

```bash
pip install langchain langgraph langsmith
```

> *If you want to use LangSmith for tracing, set the `LANGCHAIN_TRACING_V2=true` env var and pr

In [19]:
result=agent.invoke(
    {
    'messages':[{'role':'user','content':'How to setup EC2 instance'}],
    'files':skill_files,
    },
    config={'configurable':{'thread_id':'skills-demo'}},


)
print(result['messages'][-1].content)

for m in result['messages']:
    m.pretty_print()

Below is a step‑by‑step guide that covers everything you need to launch a fresh Amazon EC2 instance from scratch.  
I’ve broken it into three phases:

1. **Preparation** – IAM, networking, security, and key‑pair.  
2. **Launch** – choose AMI, instance type, storage, and start the instance.  
3. **Post‑launch** – connect, test, and (optionally) automate the process.

Feel free to copy‑paste the snippets into a terminal or a CloudFormation / Terraform script if you want to automate the whole thing.

---

## 1️⃣ Preparation

| What you need | Why it matters | Quick command / link |
|---------------|----------------|----------------------|
| **IAM user/role** | Must have `AmazonEC2FullAccess` (or the minimal set you want). | `aws iam create-user --user-name ec2-launcher`<br>`aws iam attach-user-policy --user-name ec2-launcher --policy-arn arn:aws:iam::aws:policy/AmazonEC2FullAccess` |
| **Key pair** | Enables SSH access. Keep the private key safe. | `aws ec2 create-key-pair --key-name my-e